# Keystroke Biometric Authentication: Model Training

This notebook demonstrates the process of building, training, and evaluating the K-Nearest Neighbors (KNN) model used for keystroke dynamics authentication.

### Passphrase:
> "the quick brown fox jumps over the lazy black dog"

### Features (97 total):
- **49 Dwell Times (T)**: Time each key was held down.
- **48 Flight Times (F)**: Time between consecutive key presses.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
import os

# Constants
CSV_PATH = 'bio_bio.csv'
FEATURE_COUNT = 97

## 1. Load Data
The dataset contains 97 feature columns and a `CLASS` column representing the user identity.

In [ ]:
if not os.path.exists(CSV_PATH):
    print(f"Error: {CSV_PATH} not found. Please ensure the biometric data file is in the same directory.")
else:
    df = pd.read_csv(CSV_PATH, keep_default_na=False)
    print(f"Loaded dataset with {len(df)} samples and {len(df['CLASS'].unique())} unique users.")
    display(df.head())

## 2. Preprocessing
Standardization is critical for KNN because it relies on distance calculations. We scale all features to have zero mean and unit variance.

In [ ]:
X = df.iloc[:, :FEATURE_COUNT]
y = df['CLASS']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features scaled successfully.")

## 3. Train/Test Split
We split the data to evaluate how the model performs on unseen typing samples.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

## 4. Model Building
We use a **K-Nearest Neighbors (KNN)** classifier with **Manhattan distance** and **distance-based weights**.

In [ ]:
knn = KNeighborsClassifier(
    n_neighbors=3,
    metric='manhattan',
    weights='distance'
)

knn.fit(X_train, y_train)
print("Model training complete.")

## 5. Evaluation
We check the model's accuracy and inspect the confusion matrix.

In [ ]:
y_pred = knn.predict(X_test)

print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=knn.classes_, yticklabels=knn.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## 6. Cross-Validation
To ensure stability, we perform 5-fold cross-validation.

In [ ]:
scores = cross_val_score(knn, X_scaled, y, cv=5)
print(f"Cross-validation Accuracy: {scores.mean()*100:.2f}% (+/- {scores.std()*2*100:.2f}%)")

## 7. Sample Prediction
This shows how a single feature vector is processed for verification.

In [ ]:
sample_idx = 0
sample_vector = X_scaled[sample_idx].reshape(1, -1)
actual_user = y.iloc[sample_idx]

prediction = knn.predict(sample_vector)[0]
probabilities = knn.predict_proba(sample_vector)[0]
confidence = max(probabilities)

print(f"Actual User: {actual_user}")
print(f"Predicted User: {prediction}")
print(f"Confidence Score: {confidence*100:.2f}%")